# Transfer learning: ResNet-18 to classify emotions

ResNet-18 pre-trained on ImageNet: there are 11 M parameters that already know how to recognise
edges, textures and shapes. We reuse this backbone and only relearn the end of the network
to classify the mel-spectrograms produced by `prepare_datasets.ipynb`
(folder `data/spectrograms/<emotion>/*.png`, for the Berlin EmoDB corpus).

Two classic steps:

1. **feature extraction** — the whole backbone is frozen, we train only the new
   classification layer (7 emotions);
2. **fine-tuning** — we unfreeze the last block (`layer4`) with a learning rate 10x
   smaller to specialise the high-level features on spectrograms.

The train/validation split is **by speaker** (the first 2 characters of the file name are
the speaker ID in EmoDB): otherwise the network recognises the voice and the score is
over-optimistic.

In [ ]:
# deps (kernel "Python (CNN_RO11 .venv)")
# The default PyPI torch wheels are the CUDA build: they use the GPU when there is one and
# fall back to the CPU when there isn't. On a GPU-less machine we install the slim CPU
# wheels instead, to avoid downloading ~3 GB of CUDA libraries for nothing.
#
# `pip install torch` alone is not enough to switch build: if a torch is already there,
# whichever build it is, pip considers the requirement satisfied and does nothing. So we
# look at the build actually installed (the "+cu" / "+cpu" suffix) and reinstall only if
# it does not match the machine.
import importlib.metadata as md
import shutil

gpu = shutil.which("nvidia-smi") is not None
try:
    installed = md.version("torch")
except md.PackageNotFoundError:
    installed = None
print(f"NVIDIA GPU: {gpu}   torch installed: {installed}")

wanted_cuda = gpu
have_cuda = bool(installed) and "+cu" in installed

if installed is None or wanted_cuda != have_cuda:
    print("-> (re)installing the", "CUDA" if wanted_cuda else "CPU-only", "build of torch")
    if wanted_cuda:
        %pip install -q -U --force-reinstall torch torchvision
    else:
        %pip install -q -U --force-reinstall torch torchvision --index-url https://download.pytorch.org/whl/cpu
    print("!! RESTART THE KERNEL before running the next cell !!")
else:
    print("-> the installed build matches the machine, nothing to do")

%pip install -q scikit-learn matplotlib pillow

In [ ]:
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18

# Make random operations deterministic (for reproducibility)
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

DATA = Path("data/spectrograms")
IMG_SIZE = 224                     # input size expected by ResNet-18
BATCH = 32
EPOCHS_HEAD = 8                    # step 1: head only
EPOCHS_FT = 8                      # step 2: + layer4
LR_HEAD = 1e-3
LR_FT = 1e-4
VAL_SPEAKERS = {"15", "16"}        # 2 speakers out of 10 held out for validation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| CUDA build:", torch.version.cuda)
print("device:", device, f"({torch.cuda.get_device_name(0)})" if device.type == "cuda" else "")

## 1. Image inventory and speaker split

In [ ]:
CLASSES = sorted(d.name for d in DATA.iterdir() if d.is_dir()) # labels (emotion names)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)} # dictionary mapping class names to indices

items = [(p, CLASS_TO_IDX[p.parent.name], p.stem[:2]) for p in sorted(DATA.rglob("*.png"))]
# items : list of tuples (image_path, class_index, speaker_id) for each image in the dataset
speakers = sorted({s for _, _, s in items})

train_items = [it for it in items if it[2] not in VAL_SPEAKERS]
val_items = [it for it in items if it[2] in VAL_SPEAKERS]

print("classes :", CLASSES)
print("speakers:", speakers)
print(f"train {len(train_items)} images / val {len(val_items)} images")
print("train distribution:", {CLASSES[i]: n for i, n in sorted(Counter(y for _, y, _ in train_items).items())})
print("val distribution  :", {CLASSES[i]: n for i, n in sorted(Counter(y for _, y, _ in val_items).items())})

## 2. Dataset and transforms

Images are normalised with the ImageNet statistics: this is what lets the pre-trained
weights see inputs in the distribution they know.

Augmentation: random rectangular masks only (equivalent to SpecAugment: we hide a band of
time or frequency). **No horizontal flip** — reversing time is not a plausible
spectrogram.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406] # means of the 3 channels
IMAGENET_STD = [0.229, 0.224, 0.225]  # standard deviations of the 3 channels

#Normalisation consists to do x'=(x - mean)/std for each channel of each pixel. 

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.12), ratio=(0.3, 3.3)),  # time/frequency mask: adds holes in the spectrograms to make the model more robust to missing information
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class SpectrogramDataset(Dataset):
    def __init__(self, items, tf):
        self.items, self.tf = items, tf

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, label, _ = self.items[i]
        return self.tf(Image.open(path).convert("RGB")), label


train_loader = DataLoader(SpectrogramDataset(train_items, train_tf), batch_size=BATCH,
                          shuffle=True, num_workers=0, drop_last=False)
val_loader = DataLoader(SpectrogramDataset(val_items, eval_tf), batch_size=BATCH,
                        shuffle=False, num_workers=0)

xb, yb = next(iter(train_loader))
print("batch:", tuple(xb.shape), "labels:", tuple(yb.shape))

## 3. The pre-trained model

`resnet18(weights=IMAGENET1K_V1)` downloads the 11 M parameters learned on ImageNet.
We replace the last layer (`fc`, 1000 ImageNet classes) with a 7-output layer, randomly
initialised: it is the only part that has learned nothing.

In [ ]:
def build_model():
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1) # builds resnet18 architecture with pretrained weights on ImageNet
    # the model ends with a fully connected layer (fc) that outputs 1000 classes (ImageNet classes)
    # we should replace it with a new fc layer that outputs the number of classes in our dataset (7 emotions)
    model.fc = nn.Linear(model.fc.in_features, len(CLASSES))  # 1000 -> 7
    return model.to(device)


def set_trainable(model, names):
    """Keep `requires_grad=True` only on modules whose name starts with a given prefix."""
    for name, p in model.named_parameters():
        p.requires_grad = any(name.startswith(n) for n in names)


model = build_model()
total = sum(p.numel() for p in model.parameters())
set_trainable(model, ["fc"])
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"total parameters : {total:,}")
print(f"trainable params (step 1) : {trainable:,}  ({100 * trainable / total:.2f} %)")

## 4. Training loop

The dataset is imbalanced (more "anger" than "disgust" for example): the loss is weighted by the
inverse class frequency, otherwise the model mostly learns to predict "anger".
The BatchNorm of the frozen layers is set to `eval()` so it does not drift on such a small
dataset.

In [ ]:
counts = np.bincount([y for _, y, _ in train_items], minlength=len(CLASSES))
class_weights = torch.tensor(counts.sum() / (len(CLASSES) * counts), dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
print({c: round(float(w), 2) for c, w in zip(CLASSES, class_weights)})


def freeze_bn(model):
    """BatchNorm in inference mode wherever the weights are frozen."""
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d) and not any(p.requires_grad for p in m.parameters()):
            m.eval()


def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    if train:
        freeze_bn(model)
    loss_sum, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            loss_sum += loss.item() * y.size(0)
            correct += (out.argmax(1) == y).sum().item()
            n += y.size(0)
    return loss_sum / n, correct / n


def fit(model, optimizer, epochs, history, tag):
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = run_epoch(model, train_loader, optimizer)
        va_loss, va_acc = run_epoch(model, val_loader)
        history.append({"phase": tag, "train_acc": tr_acc, "val_acc": va_acc,
                        "train_loss": tr_loss, "val_loss": va_loss})
        print(f"[{tag}] epoch {epoch:2d}  train {tr_loss:.3f}/{tr_acc:.3f}   val {va_loss:.3f}/{va_acc:.3f}")
    return history

### Step 1: frozen backbone, we only train the head

In [ ]:
history = []
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=LR_HEAD, weight_decay=1e-4)
fit(model, optimizer, EPOCHS_HEAD, history, "head")

### Step 2: fine-tuning `layer4`

We unfreeze the last convolutional block. Learning rate 10x smaller: the ImageNet
features are already good, we want to adjust them, not destroy them.

In [ ]:
set_trainable(model, ["layer4", "fc"]) # this time we also unfreeze the last convolutional block (layer4) of ResNet-18 for fine tuning
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable params (step 2) : {trainable:,}  ({100 * trainable / total:.2f} %)")

optimizer = torch.optim.AdamW([
    {"params": model.layer4.parameters(), "lr": LR_FT},
    {"params": model.fc.parameters(), "lr": LR_FT * 10},
], weight_decay=1e-4)
fit(model, optimizer, EPOCHS_FT, history, "finetune")

torch.save(model.state_dict(), "resnet18_emodb.pt")
print("weights saved to resnet18_emodb.pt")

## 5. Learning curves

In [ ]:
import matplotlib.pyplot as plt

TRAIN_C, VAL_C = "#4571C4", "#D97757"   # 2 series: legend + colours that stay distinct for colour-blind viewers
split = sum(h["phase"] == "head" for h in history)
epochs = range(1, len(history) + 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for ax, key, title in zip(axes, ["loss", "acc"], ["Loss", "Accuracy"]):
    ax.plot(epochs, [h[f"train_{key}"] for h in history], color=TRAIN_C, lw=2, label="train")
    ax.plot(epochs, [h[f"val_{key}"] for h in history], color=VAL_C, lw=2, label="validation")
    ax.axvline(split + 0.5, color="#9A9A94", lw=1, ls="--")
    ax.annotate("fine-tuning ->", (split + 0.7, ax.get_ylim()[1]), va="top", fontsize=8, color="#6B6B66")
    ax.set_title(title)
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.25, lw=0.6)
    ax.spines[["top", "right"]].set_visible(False)
axes[1].set_ylim(0, 1)
axes[0].legend(frameon=False)
plt.show()

best = max(history, key=lambda h: h["val_acc"])
print(f"best validation accuracy : {best['val_acc']:.3f}  (chance = {1 / len(CLASSES):.3f})")

## 6. Evaluation: per-class report and confusion matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for x, y in val_loader:
        y_pred += model(x.to(device)).argmax(1).cpu().tolist()
        y_true += y.tolist()

print(classification_report(y_true, y_pred, target_names=CLASSES, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASSES)), normalize="true")
fig, ax = plt.subplots(figsize=(6.5, 5.5), constrained_layout=True)
# sequential: a single hue, light -> dark (magnitude, no rainbow)
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(
    ax=ax, cmap="Blues", colorbar=False, values_format=".2f", xticks_rotation=45)
ax.set_title("Confusion matrix (validation, row-normalised)")
ax.set_xlabel("prediction")
ax.set_ylabel("ground truth")
ax.grid(False)
plt.show()

## 7. A few predictions

The images shown are the raw spectrograms (before ImageNet normalisation).

In [ ]:
rng = np.random.default_rng(SEED)
sample = [val_items[i] for i in rng.choice(len(val_items), size=8, replace=False)]

with torch.no_grad():
    batch = torch.stack([eval_tf(Image.open(p).convert("RGB")) for p, _, _ in sample]).to(device)
    probs = model(batch).softmax(1).cpu().numpy()

fig, axes = plt.subplots(2, 4, figsize=(13, 5), constrained_layout=True)
for ax, (path, label, spk), p in zip(axes.ravel(), sample, probs):
    ax.imshow(Image.open(path))
    pred = int(p.argmax())
    ok = pred == label
    ax.set_title(f"{CLASSES[pred]} {p[pred]:.0%}\ntrue : {CLASSES[label]} (spk {spk})",
                 fontsize=9, color="#2E7D4F" if ok else "#C0392B")
    ax.axis("off")
plt.show()

## 8. Going further

- **per-speaker cross-validation** (leave-one-speaker-out over the 10 speakers): with
  535 images, a single split of 2 speakers has a large variance;
- unfreeze `layer3` as well, or on the contrary compare with a ResNet-18 trained **from
  scratch** (`weights=None`) to measure what pre-training really brings;
- augmentation on the audio side (pitch/time stretch) before generating the spectrograms,
  more realistic than masking on the image;
- non-resized inputs: keep 64x300 and adapt the pooling, rather than stretching the
  spectrogram to 224x224.